# Linear Regression — Earthquake Magnitude Prediction


**Algorithm:** Linear Regression (Supervised Learning)



### Step 1: Importing all the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)

### Step 2: Data Collection

Loading the same earthquake catalogue used across the group's report, using the same file name and path convention (`Earthquake.csv`) as the rest of the team's notebooks so everything stays consistent when merged.

In [ ]:
DATA_PATH = "Earthquake.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nEvent type counts:")
print(df["type"].value_counts())

### Step 3: Data Preprocessing

Following the module's required pipeline order (Data Collection -> Preprocessing -> Feature Engineering -> Model Training), and using the exact same cleaning steps as our team's other notebooks, so all five models start from an identical, consistent dataset.

**3.1 Remove non-earthquake events.** The raw catalogue includes 3 "nuclear explosion" records picked up by the same seismic network. These are removed before training a model about earthquake magnitude.

**3.2 Handle missing values.** Several instrumentation-quality columns have a large number of missing values. As with our other notebooks, we impute with the column **median**, which is robust to outliers, rather than dropping the majority of rows.

In [ ]:
print("Rows before filtering to earthquakes only:", df.shape[0])
df = df[df["type"] == "earthquake"].copy()
print("Rows after removing non-earthquake events (e.g. nuclear explosions):", df.shape[0])

feature_cols = ["latitude", "longitude", "depth", "gap", "dmin", "rms", "nst"]

print("\nMissing values in candidate features:")
print(df[feature_cols].isnull().sum())

# Median imputation: robust to outliers, avoids dropping the majority of rows
# that columns like 'dmin' and 'gap' would otherwise cost us.
X = df[feature_cols].copy()
for col in feature_cols:
    X[col] = X[col].fillna(X[col].median())

print("\nMissing values after imputation:", X.isnull().sum().sum())

y = df.loc[X.index, "mag"]
print("\nFinal feature matrix shape:", X.shape)
print("Final target vector shape:", y.shape)

### Step 4: Feature Engineering

Unlike our team's three classifiers, Linear Regression predicts the raw, continuous `mag` column directly — there is no categorical target to engineer here. Linear Regression also does not require feature scaling the way Logistic Regression does, since Ordinary Least Squares coefficients are estimated by minimising squared error directly on the original feature scales, and scaling would only rescale the coefficients, not change the model's predictions or its R² score. No additional feature-engineering step is needed for this model.

### Step 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

### Step 6: Model Training

Linear Regression has no hyperparameters to tune in its standard form (unlike Decision Tree's `max_depth` or Random Forest's `n_estimators`) — this is itself one of its practical strengths, noted in the theory section above.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
print("Model trained. Predictions generated for", len(y_pred), "test samples.")

### Step 7: Evaluation Metrics

Since this is a **regression** task, not a classification task, we use the metrics that are actually valid for a continuous target — Accuracy, Precision, Recall, F1-score and a Confusion Matrix are mathematically undefined here, because there is no predicted class to compare against a true class.

- **R² (R-squared)** — the proportion of variance in magnitude explained by the model. Ranges up to 1.0 (perfect fit); 0 means the model explains no more than always predicting the average magnitude.
- **MAE (Mean Absolute Error)** — the average absolute difference between predicted and actual magnitude, in magnitude units.
- **RMSE (Root Mean Squared Error)** — similar to MAE, but penalises large errors more heavily, since errors are squared before averaging.

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=== Linear Regression Performance ===")
print(f"R\u00b2 Score : {r2:.4f}")
print(f"MAE       : {mae:.4f}")
print(f"RMSE      : {rmse:.4f}")

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lr_model.coef_,
}).sort_values("Coefficient", key=abs, ascending=False)

print("\nModel coefficients (sorted by absolute size):")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {lr_model.intercept_:.4f}")

### Step 8: Visualising Predicted vs. Actual Magnitude

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=15, color="#1f5fa6")

# Reference diagonal: a perfect model would put every point exactly on this line
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color="red", linestyle="--", linewidth=1.5, label="Perfect prediction")

ax.set_xlabel("Actual Magnitude")
ax.set_ylabel("Predicted Magnitude")
ax.set_title("Linear Regression: Predicted vs. Actual Magnitude")
ax.legend()
plt.tight_layout()
plt.savefig("linear_regression_predicted_vs_actual.png", dpi=150)
plt.show()

### Step 9: Making a Prediction on a New Event

In [ ]:
new_earthquake = pd.DataFrame([{
    "latitude": 27.7,     # Kathmandu, Nepal
    "longitude": 85.3,
    "depth": 10.0,
    "gap": 50.0,
    "dmin": 1.5,
    "rms": 0.7,
    "nst": 100.0,
}])

predicted_magnitude = lr_model.predict(new_earthquake)[0]

print("Input features for new event:")
print(new_earthquake)
print(f"\nPredicted magnitude: {predicted_magnitude:.2f}")

### Step 10: Save Metrics

In [ ]:
results = pd.DataFrame([{
    "Model": "Linear Regression",
    "R2": round(r2, 4),
    "MAE": round(mae, 4),
    "RMSE": round(rmse, 4),
}])
results.to_csv("linear_regression_results.csv", index=False)
print("Saved metrics table to linear_regression_results.csv")
print(results)

print("\n=== LINEAR REGRESSION COMPLETE ===")